In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf

# ── TICKER UNIVERSE ───────────────────────────────────────────────────────────
# Load the S&P 500 membership history. Each row lists every ticker that was in the index on a given date.
path = "/Users/taytematthews/Backtesting Data/S&P 500 Historical Components & Changes(01-17-2026) (1).csv"
ticker_history = pd.read_csv(path)
ticker_history["date"] = pd.to_datetime(ticker_history["date"])

# Keep only rows inside our data window (1999–2019).
# Ending at 2019 means we only use tickers that survived the full period —
# this avoids survivorship bias (e.g. ignoring companies that went bankrupt and
# were removed from the index before our backtest ends).
th_filtered = ticker_history[
    (ticker_history["date"] >= pd.Timestamp("1999-01-01")) &
    (ticker_history["date"] <= pd.Timestamp("2019-12-31"))
]

# Each row's "tickers" column is a comma-separated string like "AAPL,MSFT,...".
# We convert each row to a set, then intersect every set to find tickers that
# appear in EVERY snapshot — i.e. continuously in the S&P 500 the whole time.
ticker_sets    = th_filtered["tickers"].str.split(",").map(set)
common_tickers = set.intersection(*ticker_sets) if len(ticker_sets) > 0 else set()
len(common_tickers)

In [ ]:
import os

DATA_CACHE = "/Users/taytematthews/June Backtesting Repo/price_data_cache.pkl"

# Loading from disk is much faster than re-downloading from Yahoo Finance
# (which is slow and rate-limited). The cache stores 1999–2025 so any
# rank/backtest window we might want to test is already covered.
if os.path.exists(DATA_CACHE):
    data = pd.read_pickle(DATA_CACHE)
    print(f"Loaded cached data from {DATA_CACHE}")
else:
    # auto_adjust=False keeps raw prices; we use "Adj Close" ourselves.
    # group_by="ticker" organises columns as a (Ticker, PriceType) MultiIndex.
    data = yf.download(
        sorted(common_tickers),
        start=pd.Timestamp("1999-01-01"),
        end=pd.Timestamp("2025-12-31"),
        auto_adjust=False,
        group_by="ticker",
        progress=True,
    )
    data.to_pickle(DATA_CACHE)
    print(f"Downloaded and cached to {DATA_CACHE}")


In [ ]:
# yfinance returns a MultiIndex DataFrame where columns are (Ticker, PriceType).
# xs("Adj Close", level=1) slices across that level to produce a flat DataFrame
# where each column is one ticker — the shape we need for every calculation below.
if isinstance(data.columns, pd.MultiIndex):
    adj_close_prices = data.xs("Adj Close", axis=1, level=1).copy()
    open_prices      = data.xs("Open",      axis=1, level=1).copy()
    volumes          = data.xs("Volume",    axis=1, level=1).copy()
else:
    # Single-ticker downloads don't use a MultiIndex
    adj_close_prices = data["Adj Close"].copy()
    open_prices      = data["Open"].copy()
    volumes          = data["Volume"].copy()


In [ ]:
def compute_recoveries(adj_close_prices, start_date=None, end_date=None, analysis_start=None):
    """
    Scan each ticker for drawdown events (price >= 20% below its rolling 52-week
    high) and record how long each one took to recover by +12.5%.

    Only ONE drawdown is tracked at a time — a new event cannot begin until the
    previous one has fully recovered. This prevents a prolonged decline from being
    counted as dozens of overlapping drawdowns.
    """

    # Slice to the ranking window (local copy — doesn't modify the global DataFrame)
    if start_date or end_date:
        adj_close_prices = adj_close_prices.loc[start_date:end_date]

    # If no explicit analysis start is given, begin on the first available date
    if analysis_start is None:
        analysis_start = adj_close_prices.index[0]

    # Rolling 52-week high: the highest close in the past 252 trading days.
    # min_periods=1 means we don't need a full year of history to start —
    # useful in year 1 when we're just seeding the rolling high.
    rolling_52wk_high = adj_close_prices.rolling(window=252, min_periods=1).max()

    # True on any day the price is >= 20% below its 52-week high
    drawdown_mask = adj_close_prices <= 0.8 * rolling_52wk_high

    recovery_records = []

    for ticker in adj_close_prices.columns:
        prices = adj_close_prices[ticker]
        n = len(prices)
        i = 0  # index of the day we're currently examining

        while i < n:
            date  = prices.index[i]
            price = prices.iloc[i]

            # ── DRAWDOWN ENTRY ─────────────────────────────────────────────────
            # Two conditions must both be true before we log a drawdown:
            #   1. Price is >= 20% below its 52-week high (drawdown_mask = True)
            #   2. We're past analysis_start — the first year is only for seeding
            #      the rolling high, not for generating actual drawdown signals
            if drawdown_mask.loc[date, ticker] and date >= analysis_start:

                drawdown_price  = price
                recovery_target = drawdown_price * 1.125  # need +12.5% to count as recovered

                # ── SEARCH FOR RECOVERY ────────────────────────────────────────
                # Scan every future trading day looking for the first close that
                # hits the recovery target
                recovered    = False
                recovery_idx = None

                for j in range(i + 1, n):
                    if prices.iloc[j] >= recovery_target:
                        recovery_records.append({
                            "Ticker":           ticker,
                            "Drawdown Date":    date,
                            "Drawdown Price":   drawdown_price,
                            "Recovery Date":    prices.index[j],
                            "Recovery Price":   prices.iloc[j],
                            "Days To Recovery": j - i,  # trading days, not calendar days
                            "Return":           (prices.iloc[j] / drawdown_price) - 1,
                        })
                        recovered    = True
                        recovery_idx = j
                        break

                if recovered:
                    # Jump past the recovery date so the next drawdown we detect
                    # is guaranteed to be a fresh, independent decline
                    i = recovery_idx + 1
                    continue
                else:
                    # The stock never recovered within the window.
                    # Record it once with NaN fields so the scoring step can
                    # penalise unresolved drawdowns, then stop scanning this ticker
                    # — we can't know where a new drawdown begins until this one resolves.
                    recovery_records.append({
                        "Ticker":           ticker,
                        "Drawdown Date":    date,
                        "Drawdown Price":   drawdown_price,
                        "Recovery Date":    pd.NaT,
                        "Recovery Price":   np.nan,
                        "Days To Recovery": np.nan,
                        "Return":           np.nan,
                    })
                    break  # done with this ticker

            i += 1  # no drawdown today — advance to the next day

    recoveries = pd.DataFrame(recovery_records)

    if len(recoveries) > 0:
        recoveries = recoveries.sort_values(
            ["Drawdown Date", "Ticker"]
        ).reset_index(drop=True)

    return recoveries


In [ ]:
from scipy import stats

def compute_scores(recoveries, start_date, end_date, weights=(1, 1, 1)):
    """
    Rank tickers by how useful they are as mean-reversion candidates.
    Three metrics are combined into a single score:
      1. Drawdown count     — more drawdowns = more trading opportunities
      2. Avg recovery days  — faster recoveries = less capital tied up
      3. Year-over-year variance — low variance = consistent behavior across years

    Each metric is z-scored first so they all contribute equally regardless
    of their raw scale, then multiplied by optional weights and summed.
    """
    w_dd, w_rec, w_var = weights

    # ── Per-ticker aggregate stats ─────────────────────────────────────────────
    summary = recoveries.groupby("Ticker").agg(
        drawdown_count    = ("Ticker", "count"),
        avg_recovery_days = ("Days To Recovery", "mean"),
    ).reset_index()

    # ── Drawdown count z-score ─────────────────────────────────────────────────
    # More drawdowns = higher score (we want stocks that fall and bounce reliably)
    summary["drawdown_score_zscore"] = stats.zscore(summary["drawdown_count"])

    # ── Recovery speed z-score ────────────────────────────────────────────────
    # Faster recovery = higher score, so we negate (fewer days → more positive).
    # scipy.zscore can't handle NaN — one NaN turns the entire column to NaN.
    # Tickers whose only drawdown never recovered have avg_recovery_days = NaN;
    # substitute the column mean before z-scoring as a neutral placeholder.
    filled_recovery = summary["avg_recovery_days"].fillna(summary["avg_recovery_days"].mean())
    summary["recovery_score_zscore"] = -stats.zscore(filled_recovery)

    # ── Year-over-year consistency ─────────────────────────────────────────────
    # A ticker that draws down once a year is more reliably tradeable than one
    # that had 10 drawdowns in 2008 and zero every other year.
    # Build a table: rows = tickers, columns = calendar years, values = count.
    yearly_counts = (
        recoveries.assign(year=recoveries["Drawdown Date"].dt.year)
        .groupby(["Ticker", "year"])
        .size()
        .unstack(fill_value=0)   # years with zero drawdowns become 0, not NaN
    )
    # Variance across years: lower = more consistent = better score (negated)
    summary["drawdowns_variance"] = (
        yearly_counts.var(axis=1, ddof=0)
        .reindex(summary["Ticker"])
        .values
    )
    filled_var = summary["drawdowns_variance"].fillna(summary["drawdowns_variance"].mean())
    summary["variance_zscore"] = -stats.zscore(filled_var)

    # ── Final weighted score ───────────────────────────────────────────────────
    summary["final_score"] = (
        w_dd  * summary["drawdown_score_zscore"] +
        w_rec * summary["recovery_score_zscore"] +
        w_var * summary["variance_zscore"]
    )

    return (
        summary[[
            "Ticker",
            "drawdown_count", "avg_recovery_days", "drawdowns_variance",
            "drawdown_score_zscore", "recovery_score_zscore", "variance_zscore",
            "final_score",
        ]]
        .sort_values("final_score", ascending=False)
        .reset_index(drop=True)
    )


In [ ]:
def run_backtest(top_percentile, allocation_pct, backtest_start, backtest_end,
                 summary, rolling_52wk_high_df):
    """
    Simulate the mean-reversion strategy over a given backtest window.

    Tickers are filtered to those whose final_score is at or above `top_percentile`,
    sorted best-first. Each day we:
      1. Close any position that hit its profit target (+12.5%) or stop-loss (-20%)
      2. Mark remaining positions to market
      3. Buy any eligible ticker that is >= 20% below its 52-week high
      4. Record a daily portfolio snapshot

    `summary` and `rolling_52wk_high_df` are passed in explicitly (not read from
    globals) so run_scenario() can call this with different ranking windows.
    """

    # ── Build the eligible ticker universe ────────────────────────────────────
    # Only tickers whose score is at or above the percentile cutoff, sorted so
    # capital goes to the highest-ranked stock first when multiple signals fire
    threshold = summary["final_score"].quantile(top_percentile)
    tickers = (
        summary[summary["final_score"] >= threshold]
        .sort_values("final_score", ascending=False)["Ticker"]
        .tolist()
    )

    portfolio = {
        "cash": INITIAL_CASH,
        "open_positions": {},   # {ticker: {shares, buy_date, buy_price, entry_time_idx}}
    }
    closed_trades  = []
    equity_history = []

    for date_idx, date in enumerate(adj_close_prices.index):
        if date < backtest_start:
            continue   # skip the seed year(s) before the backtest window
        if date > backtest_end:
            break

        # ── 1. CLOSE POSITIONS THAT HIT THEIR TARGET OR STOP ──────────────────
        # list() snapshots the keys before we start, so we can safely delete
        # entries from the dict mid-loop without a RuntimeError
        for ticker in list(portfolio["open_positions"].keys()):
            current_price = adj_close_prices.loc[date, ticker]
            if pd.isna(current_price):
                continue

            position     = portfolio["open_positions"][ticker]
            price_return = (current_price / position["buy_price"]) - 1
            pnl          = (current_price - position["buy_price"]) * position["shares"]

            if price_return >= PROFIT_TARGET_PCT:
                reason = "TAKE_PROFIT"
            elif price_return <= STOP_LOSS_PCT:
                reason = "STOP_LOSS"
            else:
                continue   # still within bounds — keep holding

            # Return proceeds to cash and log the completed trade
            portfolio["cash"] += current_price * position["shares"]
            closed_trades.append({
                "Ticker":      ticker,
                "Entry Date":  position["buy_date"],
                "Entry Price": position["buy_price"],
                "Shares":      position["shares"],
                "Exit Date":   date,
                "Exit Price":  current_price,
                "PnL":         pnl,
                "PnL %":       price_return,
                "Days Held":   date_idx - position["entry_time_idx"],
                "Reason":      reason,
            })
            del portfolio["open_positions"][ticker]

        # ── 2. MARK REMAINING POSITIONS TO MARKET ─────────────────────────────
        position_equity = 0
        for ticker, position in portfolio["open_positions"].items():
            current_price = adj_close_prices.loc[date, ticker]
            if not pd.isna(current_price):
                position_equity += current_price * position["shares"]

        # ── 3. SCAN FOR NEW BUY SIGNALS ───────────────────────────────────────
        for ticker in tickers:
            if ticker in portfolio["open_positions"]:
                continue   # already holding this stock

            current_price = adj_close_prices.loc[date, ticker]
            high_52w      = rolling_52wk_high_df.loc[date, ticker]
            if pd.isna(current_price) or pd.isna(high_52w):
                continue

            drawdown_pct = (current_price / high_52w) - 1
            if drawdown_pct > -0.20:
                continue   # not in a deep enough drawdown — skip

            # Position size = fixed % of total equity (cash + current open positions).
            # Recomputing total_equity inside the loop means each new buy reflects
            # capital already committed earlier the same day.
            total_equity    = portfolio["cash"] + position_equity
            position_amount = total_equity * allocation_pct
            if portfolio["cash"] < position_amount:
                continue   # not enough free cash for a full-sized position

            shares = position_amount / current_price
            portfolio["cash"] -= position_amount
            position_equity  += position_amount   # reflect the buy immediately
            portfolio["open_positions"][ticker] = {
                "shares":         shares,
                "buy_date":       date,
                "buy_price":      current_price,
                "entry_time_idx": date_idx,
            }

        # ── 4. RECORD END-OF-DAY SNAPSHOT ─────────────────────────────────────
        equity_history.append({
            "Date":            date,
            "Cash":            portfolio["cash"],
            "Position Equity": position_equity,
            "Total Equity":    portfolio["cash"] + position_equity,
            "Open Positions":  len(portfolio["open_positions"]),
        })

    eq_df     = pd.DataFrame(equity_history)
    trades_df = pd.DataFrame(closed_trades)

    # ── PERFORMANCE SUMMARY ────────────────────────────────────────────────────
    total_return = (eq_df["Total Equity"].iloc[-1] / INITIAL_CASH) - 1
    num_years    = (eq_df["Date"].iloc[-1] - eq_df["Date"].iloc[0]).days / 365.25
    # Annualised return: compound annual growth rate.
    # Normalising by time means different-length backtests are directly comparable.
    ann_return = (1 + total_return) ** (1 / num_years) - 1 if num_years > 0 else 0
    win_rate   = (trades_df["PnL"] > 0).mean() if len(trades_df) > 0 else float("nan")

    return {
        "percentile":        top_percentile,
        "allocation_pct":    allocation_pct,
        "ticker_count":      len(tickers),
        "final_equity":      eq_df["Total Equity"].iloc[-1],
        "total_return":      total_return,
        "annualized_return": ann_return,
        "num_trades":        len(trades_df),
        "win_rate":          win_rate,
        "equity_curve":      eq_df,
        "closed_trades":     trades_df,
    }


In [ ]:
# ── BACKTEST CONSTANTS ────────────────────────────────────────────────────────
INITIAL_CASH      = 100_000   # starting portfolio value ($)
PROFIT_TARGET_PCT =   0.125   # close a long when price is up +12.5% from entry
STOP_LOSS_PCT     =  -0.20    # close a long when price is down -20% from entry

# Parameter sweep ranges — one backtest run per (percentile × allocation) combination
percentiles = [0, 0.25, 0.5, 0.75]              # minimum rank percentile to enter a trade
allocations = [0.01, 0.02, 0.03, 0.04, 0.05,    # position size as fraction of total equity
               0.06, 0.07, 0.08, 0.09, 0.10]

# Pre-compute the rolling 52-week high once and reuse it across every run_backtest() call.
# Computing it inside each call would give identical results but waste time.
rolling_52wk_high = adj_close_prices.rolling(window=252, min_periods=1).max()


In [ ]:
def run_scenario(rank_start, rank_end, backtest_start, backtest_end,
                 percentiles=percentiles, allocations=allocations,
                 label=None):
    """
    Run the full pipeline — rank, score, then sweep — for any pair of date windows.

    rank_start / rank_end       : tickers are scored on their drawdown behavior
                                  over THIS period
    backtest_start / backtest_end : the scored tickers are then traded over THIS period

    Separating the two windows lets you test whether a ranking trained on one
    market regime still works in a different one (e.g. rank on 2010-2020, trade 2021-2025).
    """
    rank_start     = pd.Timestamp(rank_start)
    rank_end       = pd.Timestamp(rank_end)
    backtest_start = pd.Timestamp(backtest_start)
    backtest_end   = pd.Timestamp(backtest_end)

    # Year 1 of the rank window is used only to seed the rolling 52-week high.
    # Drawdown signals are only counted from year 2 onward (analysis_start).
    analysis_start = rank_start + pd.DateOffset(years=1)

    # Step 1: find every drawdown/recovery event in the rank window
    scenario_recoveries = compute_recoveries(
        adj_close_prices,
        start_date=rank_start,
        end_date=rank_end,
        analysis_start=analysis_start,
    )

    # Step 2: score and rank tickers by their drawdown/recovery behavior
    scenario_summary = compute_scores(scenario_recoveries, rank_start, rank_end)

    # Step 3: one backtest for every (percentile, allocation) combination
    scenario_results = [
        run_backtest(pct, alloc, backtest_start, backtest_end,
                     scenario_summary, rolling_52wk_high)
        for pct in percentiles
        for alloc in allocations
    ]

    return {
        "label":         label or (f"Rank {rank_start.date()}–{rank_end.date()}  →  "
                                   f"Backtest {backtest_start.date()}–{backtest_end.date()}"),
        "rank_start":    rank_start,
        "rank_end":      rank_end,
        "backtest_start": backtest_start,
        "backtest_end":   backtest_end,
        "recoveries":    scenario_recoveries,
        "summary":       scenario_summary,
        "results":       scenario_results,
    }


In [ ]:
####################################################
##  CHANGE THESE DATES, THEN RE-RUN THIS CELL   ##
####################################################

# Ranking window: tickers are scored on their drawdown/recovery history here.
# Year 1 seeds the rolling 52-week high; drawdown signals count from year 2 on.
RANK_START = pd.Timestamp("2010-01-01")
RANK_END   = pd.Timestamp("2020-12-31")

# Backtest window: the period during which ranked tickers are actually traded.
BACKTEST_START = pd.Timestamp("2021-01-01")
BACKTEST_END   = pd.Timestamp("2025-12-31")

####################################################

print(f"Ranking  {RANK_START.date()} → {RANK_END.date()}")
print(f"Backtest {BACKTEST_START.date()} → {BACKTEST_END.date()}")

scenario_A = run_scenario(
    rank_start     = RANK_START,
    rank_end       = RANK_END,
    backtest_start = BACKTEST_START,
    backtest_end   = BACKTEST_END,
)

# ── Summary table ─────────────────────────────────────────────────────────────
scenario_A_df = pd.DataFrame([
    {
        "Percentile":          r["percentile"],
        "Position Size":       f"{int(r['allocation_pct']*100)}%",
        "Final Equity":        r["final_equity"],
        "Total Return %":      r["total_return"] * 100,
        "Annualized Return %": r["annualized_return"] * 100,
        "Trades":              r["num_trades"],
        "Win Rate %":          r["win_rate"] * 100 if not pd.isna(r["win_rate"]) else float("nan"),
    }
    for r in scenario_A["results"]
]).sort_values(["Percentile", "Position Size"]).reset_index(drop=True)

print(scenario_A_df.to_string(index=False))

# ── Annualized-return heatmap ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

pct_vals   = sorted({r["percentile"]     for r in scenario_A["results"]})
alloc_vals = sorted({r["allocation_pct"] for r in scenario_A["results"]})

# Build a 2-D matrix: rows = percentile filters, columns = position sizes
matrix = np.array([
    [
        next(
            (r["annualized_return"] * 100
             for r in scenario_A["results"]
             if r["percentile"] == pct and abs(r["allocation_pct"] - alloc) < 1e-9),
            float("nan"),
        )
        for alloc in alloc_vals
    ]
    for pct in pct_vals
])

row_labels = [f"p{int(p*100)}" for p in pct_vals]
col_labels = [f"{int(a*100)}%"  for a in alloc_vals]

fig, ax = plt.subplots(figsize=(max(6, len(col_labels) * 1.1), len(row_labels) + 1.5))
im = ax.imshow(matrix, cmap="YlGn", aspect="auto",
               vmin=np.nanmin(matrix), vmax=np.nanmax(matrix))

ax.set_xticks(range(len(col_labels)));  ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(range(len(row_labels)));  ax.set_yticklabels(row_labels, fontsize=10)
ax.set_xlabel("Position Size (% of equity)", fontsize=11)
ax.set_ylabel("Ticker Rank Percentile Filter", fontsize=11)
ax.set_title(f"Annualized Return % — {scenario_A['label']}",
             fontsize=13, fontweight="bold", pad=12)

# Annotate each cell; use white text on dark cells and black on light cells
norm_range = np.nanmax(matrix) - np.nanmin(matrix)
for i in range(len(row_labels)):
    for j in range(len(col_labels)):
        val = matrix[i, j]
        if not np.isnan(val):
            brightness = (val - np.nanmin(matrix)) / (norm_range + 1e-9)
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center", fontsize=9,
                    color="white" if brightness > 0.6 else "black", fontweight="bold")

plt.colorbar(im, ax=ax, label="Annualized Return (%)", shrink=0.8)
plt.tight_layout()
plt.show()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── RECOVERIES VIEWER ─────────────────────────────────────────────────────────
# Shows every drawdown event recorded during the ranking window.
# Select a specific ticker to filter, or leave on "All" to see everything.

_rec_df     = scenario_A["recoveries"].copy()
_rec_select = widgets.Dropdown(
    options=["All"] + sorted(_rec_df["Ticker"].unique().tolist()),
    value="All",
    description="Ticker:",
    layout=widgets.Layout(width="200px"),
)
_rec_out = widgets.Output()

def _show_recoveries(_=None):
    ticker = _rec_select.value
    df     = _rec_df if ticker == "All" else _rec_df[_rec_df["Ticker"] == ticker]
    with _rec_out:
        clear_output(wait=True)
        # Summary line
        print(f"{len(df)} records  |  "
              f"Recovered: {df['Recovery Date'].notna().sum()}  |  "
              f"Never recovered: {df['Recovery Date'].isna().sum()}  |  "
              f"Avg days to recovery: {df['Days To Recovery'].mean():.0f}")
        # Format columns for display
        fmt = df.copy().reset_index(drop=True)
        fmt["Drawdown Price"]   = fmt["Drawdown Price"].map("${:.2f}".format)
        fmt["Recovery Price"]   = fmt["Recovery Price"].apply(
            lambda x: f"${x:.2f}" if pd.notna(x) else "—")
        fmt["Days To Recovery"] = fmt["Days To Recovery"].apply(
            lambda x: f"{x:.0f}" if pd.notna(x) else "—")
        fmt["Return"]           = fmt["Return"].apply(
            lambda x: f"{x:.1%}" if pd.notna(x) else "—")
        display(fmt)

_rec_select.observe(_show_recoveries, names="value")
display(_rec_select, _rec_out)
_show_recoveries()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── TICKER SCORES VIEWER ──────────────────────────────────────────────────────
# Shows the full ranking table: raw metrics, z-scores, and final score.
# Use the controls to sort by any column or filter to a specific ticker.

_sum_df = scenario_A["summary"].copy()

_sort_select = widgets.Dropdown(
    options=[
        ("Final Score",          "final_score"),
        ("Drawdown Count",       "drawdown_count"),
        ("Avg Days to Recovery", "avg_recovery_days"),
        ("Drawdown Variance",    "drawdowns_variance"),
    ],
    value="final_score",
    description="Sort by:",
    layout=widgets.Layout(width="260px"),
)
_sort_asc = widgets.ToggleButton(
    value=False, description="Ascending",
    layout=widgets.Layout(width="110px"),
)
_ticker_filter = widgets.Text(
    placeholder="Filter ticker…",
    description="Ticker:",
    layout=widgets.Layout(width="200px"),
)
_sum_out = widgets.Output()

def _show_summary(_=None):
    df   = _sum_df.copy()
    filt = _ticker_filter.value.strip().upper()
    if filt:
        # Partial case-insensitive match (e.g. "AA" matches "AAPL")
        df = df[df["Ticker"].str.contains(filt, na=False)]
    df = df.sort_values(_sort_select.value, ascending=_sort_asc.value).reset_index(drop=True)
    with _sum_out:
        clear_output(wait=True)
        print(f"{len(df)} tickers")
        fmt = df.copy()
        fmt["avg_recovery_days"]     = fmt["avg_recovery_days"].apply(
            lambda x: f"{x:.1f}" if pd.notna(x) else "—")
        fmt["drawdowns_variance"]    = fmt["drawdowns_variance"].apply(
            lambda x: f"{x:.2f}" if pd.notna(x) else "—")
        fmt["drawdown_score_zscore"] = fmt["drawdown_score_zscore"].map("{:.2f}".format)
        fmt["recovery_score_zscore"] = fmt["recovery_score_zscore"].map("{:.2f}".format)
        fmt["variance_zscore"]       = fmt["variance_zscore"].map("{:.2f}".format)
        fmt["final_score"]           = fmt["final_score"].map("{:.2f}".format)
        display(fmt)

for w in (_sort_select, _sort_asc, _ticker_filter):
    w.observe(_show_summary, names="value")

display(widgets.HBox([_sort_select, _sort_asc, _ticker_filter]), _sum_out)
_show_summary()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Download S&P 500 for the full data range so it covers any backtest window
sp500_data = yf.download(
    "^GSPC",
    start=pd.Timestamp("1999-01-01"),
    end=pd.Timestamp("2025-12-31"),
    progress=False,
)

def _extract_close(df):
    """Handle variations in column naming across yfinance versions."""
    if "Adj Close" in df.columns: return df["Adj Close"]
    if "Close"     in df.columns: return df["Close"]
    if isinstance(df.columns, pd.MultiIndex):
        for col in df.columns:
            if "Adj" in str(col) or "Close" in str(col): return df[col]
    return df.iloc[:, 0]

sp500_prices = _extract_close(sp500_data).squeeze()


def plot_equity_vs_sp500(result, title=None):
    """Plot one backtest result's equity curve against S&P 500 buy & hold."""
    eq = result["equity_curve"].copy()
    eq["Date"] = pd.to_datetime(eq["Date"])
    eq = eq.set_index("Date")

    # Trim S&P 500 to the same date range as the backtest
    sp = sp500_prices.loc[eq.index[0] : eq.index[-1]].copy()

    # Normalise both series to start at INITIAL_CASH for a fair comparison
    eq_norm = eq["Total Equity"] / eq["Total Equity"].iloc[0] * INITIAL_CASH
    sp_norm = sp / sp.iloc[0] * INITIAL_CASH

    pct_str   = f"p{int(result['percentile'] * 100)}"
    alloc_str = f"{int(result['allocation_pct'] * 100)}%"
    ann       = result["annualized_return"] * 100
    wr        = result["win_rate"] * 100 if not pd.isna(result["win_rate"]) else float("nan")
    sp_years  = (sp.index[-1] - sp.index[0]).days / 365.25
    sp_ann    = ((sp_norm.iloc[-1] / sp_norm.iloc[0]) ** (1 / sp_years) - 1) * 100 if sp_years > 0 else float("nan")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(eq_norm.index, eq_norm.values, color="#2563EB", linewidth=1.8,
            label=f"Strategy ({pct_str}, {alloc_str}) — {ann:.1f}% ann., {wr:.1f}% win rate")
    ax.plot(sp_norm.index, sp_norm.values, color="#9CA3AF", linewidth=1.4, linestyle="--",
            label=f"S&P 500 buy & hold — {sp_ann:.1f}% ann.")

    ax.set_title(title or f"{scenario_A['label']}  |  {pct_str}, {alloc_str}",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel(f"Portfolio Value (starting ${INITIAL_CASH:,.0f})")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.legend(fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── SHARED COMBO LOOKUP ────────────────────────────────────────────────────────
# Build sorted labels ("p0 / 1%", "p0 / 2%", …) and a dict mapping each label
# back to its result dict. Both are reused by the cash-curve and trades cells below.
_combos      = sorted(scenario_A["results"], key=lambda r: (r["percentile"], r["allocation_pct"]))
combo_labels = [f"p{int(r['percentile']*100)} / {int(r['allocation_pct']*100)}%" for r in _combos]
combo_map    = dict(zip(combo_labels, _combos))

# ── EQUITY CURVES (multi-overlay) ─────────────────────────────────────────────
# Ctrl/Cmd+click to select multiple combinations; all are overlaid on one chart
# with the S&P 500 as a benchmark.
_eq_select = widgets.SelectMultiple(
    options=combo_labels, value=[combo_labels[0]],
    layout=widgets.Layout(height="290px", width="190px"),
)
_eq_out = widgets.Output()

def _draw_equity(_=None):
    sel = list(_eq_select.value)
    with _eq_out:
        clear_output(wait=True)
        if not sel:
            return
        fig, ax = plt.subplots(figsize=(12, 5))
        colors = plt.cm.tab10.colors
        for i, lbl in enumerate(sel):
            r    = combo_map[lbl]
            eq   = r["equity_curve"].set_index(pd.to_datetime(r["equity_curve"]["Date"]))
            norm = eq["Total Equity"] / eq["Total Equity"].iloc[0] * INITIAL_CASH
            ann  = r["annualized_return"] * 100
            wr   = r["win_rate"] * 100 if not pd.isna(r["win_rate"]) else float("nan")
            ax.plot(norm.index, norm, color=colors[i % 10], lw=1.8,
                    label=f"{lbl} — {ann:.1f}% ann., {wr:.1f}% WR")
        # S&P 500 aligned to the backtest date window
        eq0  = combo_map[sel[0]]["equity_curve"]
        sp   = sp500_prices.loc[pd.to_datetime(eq0["Date"].iloc[0]) :
                                pd.to_datetime(eq0["Date"].iloc[-1])]
        spn  = sp / sp.iloc[0] * INITIAL_CASH
        sp_y = (sp.index[-1] - sp.index[0]).days / 365.25
        sp_a = ((spn.iloc[-1] / spn.iloc[0]) ** (1 / sp_y) - 1) * 100 if sp_y > 0 else float("nan")
        ax.plot(spn.index, spn, color="#9CA3AF", lw=1.4, ls="--",
                label=f"S&P 500 — {sp_a:.1f}% ann.")
        ax.set_title(f"Equity Curves — {scenario_A['label']}", fontweight="bold")
        ax.set_ylabel(f"Portfolio Value (starting ${INITIAL_CASH:,.0f})")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.legend(fontsize=8, ncol=max(1, len(sel) // 6))
        ax.grid(axis="y", ls="--", alpha=0.4)
        plt.tight_layout()
        plt.show()

_eq_select.observe(_draw_equity, names="value")
display(widgets.HBox([
    widgets.VBox([widgets.Label("Ctrl/Cmd+click to select multiple"), _eq_select]),
    _eq_out,
]))
_draw_equity()


In [ ]:
# ── CASH CURVES (multi-overlay) ───────────────────────────────────────────────
# Same structure as the equity curve cell above, but plots uninvested cash balance
# instead of total portfolio value. The S&P 500 line shows what the same capital
# could have earned if left fully invested — useful for seeing when the strategy
# sits on the sidelines vs. when it's fully deployed.
_cash_select = widgets.SelectMultiple(
    options=combo_labels, value=[combo_labels[0]],
    layout=widgets.Layout(height="290px", width="190px"),
)
_cash_out = widgets.Output()

def _draw_cash(_=None):
    sel = list(_cash_select.value)
    with _cash_out:
        clear_output(wait=True)
        if not sel:
            return
        fig, ax = plt.subplots(figsize=(12, 5))
        colors = plt.cm.tab10.colors
        for i, lbl in enumerate(sel):
            r  = combo_map[lbl]
            eq = r["equity_curve"].set_index(pd.to_datetime(r["equity_curve"]["Date"]))
            ax.plot(eq.index, eq["Cash"], color=colors[i % 10], lw=1.8, label=lbl)
        ax.set_title(f"Cash Curves — {scenario_A['label']}", fontweight="bold")
        ax.set_ylabel("Uninvested Cash ($)")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.legend(fontsize=8, ncol=max(1, len(sel) // 6))
        ax.grid(axis="y", ls="--", alpha=0.4)
        plt.tight_layout()
        plt.show()

_cash_select.observe(_draw_cash, names="value")
display(widgets.HBox([
    widgets.VBox([widgets.Label("Ctrl/Cmd+click to select multiple"), _cash_select]),
    _cash_out,
]))
_draw_cash()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── TRADES TABLE ──────────────────────────────────────────────────────────────
# Pick any parameter combination to inspect its full list of closed trades,
# with a summary line showing overall win rate and P&L.
_trades_select = widgets.Dropdown(
    options=combo_labels, value=combo_labels[0],
    description="Combo:",
    layout=widgets.Layout(width="260px"),
)
_trades_out = widgets.Output()

def _show_trades(_=None):
    r      = combo_map[_trades_select.value]
    trades = r["closed_trades"].copy()
    with _trades_out:
        clear_output(wait=True)
        if len(trades) == 0:
            print("No closed trades for this combination.")
            return
        wins = (trades["PnL"] > 0).sum()
        print(f"{len(trades)} trades  |  "
              f"Win rate: {wins / len(trades) * 100:.1f}%  |  "
              f"Avg PnL: ${trades['PnL'].mean():,.0f}  |  "
              f"Total PnL: ${trades['PnL'].sum():,.0f}")
        # Format dollar and percent columns for readability
        fmt = trades.copy()
        fmt["Entry Price"] = fmt["Entry Price"].map("${:.2f}".format)
        fmt["Exit Price"]  = fmt["Exit Price"].map("${:.2f}".format)
        fmt["PnL"]         = fmt["PnL"].map("${:,.2f}".format)
        fmt["PnL %"]       = fmt["PnL %"].map("{:.1%}".format)
        display(fmt.reset_index(drop=True))

_trades_select.observe(_show_trades, names="value")
display(_trades_select, _trades_out)
_show_trades()
